# ringdownanalysis Quick Start

This notebook is the fastest way to try the library: generate a synthetic ring-down,
look at the time series, run `RingDownAnalyzer`, and inspect the estimates visually.

The workflow mirrors the README quickstart but uses the full analysis pipeline
(`analyze_array`) instead of calling individual estimators by hand. The same API
accepts NumPy arrays, pandas objects, or files — see
[`0.2_drifting-resonators.ipynb`](0.2_drifting-resonators.ipynb) for real measurement data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ringdownanalysis import RingDownAnalyzer, RingDownSignal, plot_q_envelope_overlay

In [ ]:
# Apply consistent plotting style
from ringdownanalysis import plots

plots.apply_plotting_style()

## 1. Generate a test signal

Use `RingDownSignal` to create a noisy exponentially decaying sinusoid:
$$x(t) = A_0 \exp(-t/\tau) \cos(2\pi f_0 t + \phi_0) + \text{noise}$$

In [ ]:
# Signal parameters
f0 = 5.0  # Hz
fs = 1000.0  # Hz
N = 100000  # samples
A0 = 0.1
snr_db = 50.0  # dB
Q = 500.0  # quality factor

# Generate noisy ring-down signal
rng = np.random.default_rng(42)
signal = RingDownSignal(f0=f0, fs=fs, N=N, A0=A0, snr_db=snr_db, Q=Q)
t, data, phi0 = signal.generate(rng=rng)

print(f"Generated {N} samples at {fs} Hz ({N / fs:.1f} s total)")
print(f"True: f0={f0} Hz, tau={signal.tau:.4f} s, Q={Q}")
print(f"Noise sigma: {signal.sigma:.6f}")

## 2. Inspect the time series

Before fitting anything, plot the raw phase. A ring-down should look like a damped
oscillation whose amplitude shrinks over time.

In [ ]:
step = max(1, len(t) // 20_000)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t[::step], data[::step], "b-", alpha=0.7, linewidth=0.8)
ax.axvline(signal.tau, color="C2", linestyle="--", label=f"true τ = {signal.tau:.2f} s")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Phase (cycles)")
ax.set_title("Synthetic ring-down — full record")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Zoom: first few cycles and the first decay time
zoom_end = min(3.0 * signal.tau, t[-1])
mask = t <= zoom_end

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(t[mask], data[mask], "b-", alpha=0.8, linewidth=0.9)
ax.axvline(signal.tau, color="C2", linestyle="--", label=f"τ = {signal.tau:.2f} s")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Phase (cycles)")
ax.set_title(f"First {zoom_end:.1f} s")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
n_zoom = np.arange(mask.sum())
ax.plot(n_zoom, data[mask], "b-", alpha=0.8, linewidth=0.9)
ax.axvline(int(signal.tau * fs), color="C2", linestyle="--", label=f"τ ≈ sample {int(signal.tau * fs)}")
ax.set_xlabel("Sample index")
ax.set_ylabel("Phase (cycles)")
ax.set_title("Same window vs. sample index")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Run the analyzer

`RingDownAnalyzer.analyze_array()` runs the full pipeline: crop selection, frequency
estimation, Q diagnostics, and `Q_selected` — the estimate the library recommends
for this record.

In [ ]:
analyzer = RingDownAnalyzer()
result = analyzer.analyze_array(t=t, data=data)

print("Headline result:")
print(
    f"  Q_selected = {result['Q_selected']:.4f}"
    f"  (source: {result['Q_selected_source']}, status: {result['Q_selected_status']})"
)
print(f"  tau_est = {result['tau_est']:.6f} s  (true: {signal.tau:.6f} s)")
print(f"  Cropped to T_crop = {result['T_crop']:.2f} s ({result['N_crop']} samples)")

print("\nFrequency estimates:")
print(f"  f_nls = {result['f_nls']:.9f} Hz  (true: {f0} Hz)")
print(f"  f_dft = {result['f_dft']:.9f} Hz")

print("\nQ diagnostics (all should agree on this clean synthetic):")
print(f"  Q_demod     = {result['Q_demod']:.4f}")
print(f"  Q_nls       = {result['Q_nls']:.4f}")
print(f"  Q_dft       = {result['Q_dft']:.4f}")
print(f"  Q_profile   = {result['Q_profile']:.4f}")
print(f"  Q_envelope  = {result['Q_envelope']:.4f}")
print(f"  Plug-in frequency uncertainty std: {result['plugin_crlb_std_f']:.6e} Hz")

## 4. Other input formats

The same analyzer accepts `(data, fs)` when time is uniform, or pandas Series and
DataFrames with named columns.

In [ ]:
result_fs = analyzer.analyze_array(data=data, fs=fs)
series = pd.Series(data)
result_series = analyzer.analyze_array(t=t, data=series)
df = pd.DataFrame({"time_s": t, "phase_cycles": data})
result_df = analyzer.analyze_array(
    data=df,
    time_col="time_s",
    data_col="phase_cycles",
)

for label, r in [
    ("(data, fs)", result_fs),
    ("pandas Series", result_series),
    ("pandas DataFrame", result_df),
]:
    print(
        f"{label:>18}: Q_selected={r['Q_selected']:.4f}, "
        f"f_nls={r['f_nls']:.9f} Hz, tau={r['tau_est']:.4f} s"
    )

## 5. Visualize the analysis

Compare the analyzed crop to the full record, then overlay the measured decay
envelope with the fitted exponential. On real data the envelope plot is also a
sanity check: a candidate Q whose slope disagrees with the measured envelope is
flagged by the library.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

step = max(1, len(result["t"]) // 20_000)
step_crop = max(1, len(result["t_crop"]) // 20_000)

ax.plot(result["t"][::step], result["data"][::step], "b-", alpha=0.5, label="Full data")
ax.plot(
    result["t_crop"][::step_crop],
    result["data_cropped"][::step_crop],
    "r-",
    alpha=0.8,
    label=f"Analyzed crop (≤ {result['T_crop']:.1f} s)",
)
ax.axvline(result["tau_est"], color="C2", linestyle="--", label=f"τ_est = {result['tau_est']:.2f} s")
ax.axvline(result["T_crop"], color="C3", linestyle=":", label=f"T_crop = {result['T_crop']:.1f} s")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Phase (cycles)")
ax.set_title("Full record vs. analyzed crop")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
plot_q_envelope_overlay(ax, result, q_source="best")
ax.set_title(
    f"Decay envelope — Q_selected = {result['Q_selected']:.1f} "
    f"(true Q = {Q:.1f})"
)
plt.tight_layout()
plt.show()

In [ ]:
estimators = {
    "true Q": Q,
    "Q_selected": result["Q_selected"],
    "Q_demod": result["Q_demod"],
    "Q_nls": result["Q_nls"],
    "Q_dft": result["Q_dft"],
    "Q_profile": result["Q_profile"],
    "Q_envelope": result["Q_envelope"],
}
labels = list(estimators.keys())
values = np.array(list(estimators.values()), dtype=float)
rel_error_pct = 100.0 * (values - Q) / Q

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

colors = ["C2" if label == "true Q" else "C0" for label in labels]
axes[0].bar(labels, values, color=colors, alpha=0.85)
axes[0].axhline(Q, color="C2", linestyle="--", linewidth=1.2, label="true Q")
axes[0].set_ylabel("Q")
axes[0].set_title("Q estimates vs. truth")
axes[0].tick_params(axis="x", rotation=35)
axes[0].grid(True, axis="y", alpha=0.3)
axes[0].legend()

axes[1].bar(labels, rel_error_pct, color=colors, alpha=0.85)
axes[1].axhline(0.0, color="C2", linestyle="--", linewidth=1.2)
axes[1].set_ylabel("Relative error (%)")
axes[1].set_title("Percent deviation from true Q")
axes[1].tick_params(axis="x", rotation=35)
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### Next steps

- Real measurement files: [`0.2_drifting-resonators.ipynb`](0.2_drifting-resonators.ipynb)
- Batch processing: [`0.1_batch-analysis.ipynb`](0.1_batch-analysis.ipynb)
- Profile-likelihood Q: [`0.3_profile-likelihood-q.ipynb`](0.3_profile-likelihood-q.ipynb)
- Segmented demodulation details: [`20260819_EDU_SegmentedDemod_Estimator_Demo.ipynb`](20260819_EDU_SegmentedDemod_Estimator_Demo.ipynb)